In [1]:
import pandas as pd
from keplergl import KeplerGl

In [10]:
# Replace with your dataset file name
bike_trips = pd.read_csv("citibike_2022.csv")

/var/folders/z7/kqs18yh15zq9sxssld03yyvh0000gn/T/ipykernel_17084/3323846164.py:2: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  bike_trips = pd.read_csv("citibike_2022.csv")


In [11]:
# Add helper column to count each trip
bike_trips['value'] = 1

In [12]:
# 2. Group by Start & End Stations (with Coordinates)

trip_summary = (
    bike_trips
    .groupby([
        'start_station_name',
        'end_station_name',
        'start_lat',
        'start_lng',
        'end_lat',
        'end_lng'
    ], as_index=False)
    .agg({'value': 'count'})           # Count trips
    .rename(columns={'value': 'trip_count'})
)

# Save grouped data
trip_summary.to_csv("trip_summary.csv", index=False)

In [13]:
# 3. Kepler.gl Configuration

config = {
    "version": "v1",
    "config": {
        "visState": {
            "filters": [
                {
                    "dataId": ["Trips Between Stations"],
                    "id": "trip_count_filter",
                    "name": ["trip_count"],
                    "type": "range",
                    "value": [trip_summary['trip_count'].min(), trip_summary['trip_count'].max()],
                    "enlarged": True
                }
            ],
            "layers": [
                {
                    "id": "start_points",
                    "type": "point",
                    "config": {
                        "dataId": "Trips Between Stations",
                        "label": "Start Points",
                        "color": [0, 0, 255],  # Blue points
                        "columns": {
                            "lat": "start_lat",
                            "lng": "start_lng"
                        },
                        "isVisible": True,
                        "visConfig": {"radius": 10}
                    }
                },
                {
                    "id": "end_points",
                    "type": "point",
                    "config": {
                        "dataId": "Trips Between Stations",
                        "label": "End Points",
                        "color": [0, 255, 0],  # Green points
                        "columns": {
                            "lat": "end_lat",
                            "lng": "end_lng"
                        },
                        "isVisible": True,
                        "visConfig": {"radius": 10}
                    }
                },
                {
                    "id": "trip_arcs",
                    "type": "arc",
                    "config": {
                        "dataId": "Trips Between Stations",
                        "label": "Trip Arcs",
                        "color": [255, 0, 0],  # Red arcs
                        "columns": {
                            "lat0": "start_lat",
                            "lng0": "start_lng",
                            "lat1": "end_lat",
                            "lng1": "end_lng"
                        },
                        "isVisible": True,
                        "visConfig": {"thickness": 2}
                    }
                }
            ],
            "interactionConfig": {
                "tooltip": {
                    "enabled": True,
                    "fieldsToShow": {
                        "Trips Between Stations": [
                            "trip_count",
                            "start_station_name",
                            "end_station_name"
                        ]
                    }
                }
            },
            "layerBlending": "normal",
        },
        "mapState": {
            "latitude": 40.75,
            "longitude": -73.98,
            "zoom": 11,
            "pitch": 0,
            "bearing": 0
        },
        "mapStyle": {"styleType": "dark"}
    }
}

In [14]:
# 4. Display Map
# -------------------------
map_1 = KeplerGl(height=600)
map_1.add_data(trip_summary, "Trips Between Stations")
map_1

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


KeplerGl(data={'Trips Between Stations': {'index': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, …

In [15]:
# 5. Save HTML Map

map_1.save_to_html(file_name='nyc_trip_map.html', config=config)
print("trip_summary.csv and nyc_trip_map.html have been saved.")

Map saved to nyc_trip_map.html!
trip_summary.csv and nyc_trip_map.html have been saved.


## Map Customization

For the map visualization, I added two layers in **Kepler.gl**:

1. **Arc Layer**
   - **Source Latitude/Longitude**: `start_lat`, `start_lng`
   - **Target Latitude/Longitude**: `end_lat`, `end_lng`
   - **Color**: Red (to highlight movement)
   - **Width**: Proportional to `trip_count` (busier routes appear thicker)

2. **Point Layer**
   - **Location**: `start_lat`, `start_lng`
   - **Color**: Blue (to visually separate fixed stations from moving arcs)
   - **Size**: Constant or proportional to `trip_count` for indicating station usage intensity

**Design Rationale:**
- Red arcs stand out against blue points, making it easy to distinguish trips from station locations.
- Thicker arcs emphasize high-frequency routes, making popular routes visually prominent.
- Blue was chosen for stations because it is visually calm and contrasts well with red arcs.

### Filter for Common Trips

I added a filter on `trip_count` to focus only on frequently used trips:
- Moving the slider hides low-frequency trips and highlights core commuting patterns.
- This allows us to identify the busiest connections and focus on key travel corridors.


### Insights

From the filtered data:
- The most frequent trips occur around **Midtown Manhattan** and **Brooklyn–Manhattan connectors**.
- Central business areas such as **Penn Station** and **Grand Central** exhibit high commuting activity.
- Waterfront park areas also show significant activity, indicating recreational and tourism-based trips.
These patterns suggest a mix of **commuting behavior** and **leisure riding hotspots**, with infrastructure and geography strongly influencing usage.